In [ ]:
!pip install -q langgraph langchain-groq langchain-core pydantic

This cell installs the necessary Python libraries: `langgraph`, `langchain-groq`, `langchain-core`, and `pydantic`.

In [ ]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

This cell imports `userdata` from `google.colab` to access secrets and sets the `GROQ_API_KEY` environment variable using a key stored in Colab's secrets manager.

In [ ]:
from typing import TypedDict, Optional, List

class AgentState(TypedDict):
    query: str
    intent: Optional[str]
    confidence: Optional[float]
    sentiment: Optional[str]
    response: Optional[str]
    escalated: bool
    conversation_history: List[str]

This cell defines `AgentState` using `TypedDict` to represent the state of the conversational agent, including query, intent, confidence, sentiment, response, escalation status, and conversation history.

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

This cell initializes the `ChatGroq` language model with `llama-3.3-70b-versatile` and a temperature of 0 for deterministic output.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class ClassificationResult(BaseModel):
    intent: Literal["billing", "technical", "faq", "escalation"] = Field(
        description="The category of the query"
    )
    confidence: float = Field(description="Confidence score between 0 and 1")
    sentiment: Literal["positive", "neutral", "negative"] = Field(
        description="Tone or mood of the user"
    )
    reasoning: str = Field(description="A one-line reason for choosing this category")

This cell defines a Pydantic model `ClassificationResult` to structure the output of the query classification, including intent, confidence, sentiment, and reasoning.

In [ ]:
structured_llm = llm.with_structured_output(ClassificationResult)

def classifier_node(state: AgentState) -> AgentState:
    result = structured_llm.invoke(
        f"""Classify this customer support query.

Query: {state['query']}

Rules:
- If user mentions "manager", "refund now", "legal", "cancel subscription" → escalation
- billing = payments, invoices, subscriptions, refunds
- technical = bugs, errors, how-to, troubleshooting
- faq = general info questions
"""
    )

    state["intent"] = result.intent
    state["confidence"] = result.confidence
    state["sentiment"] = result.sentiment

    return state

This cell creates a `structured_llm` by wrapping the `llm` with the `ClassificationResult` schema. It also defines the `classifier_node` function, which uses this structured LLM to classify customer queries based on predefined rules and update the agent's state with the intent, confidence, and sentiment.

In [ ]:
test_state = {"query": "My refund is not processed yet, I want to talk to manager", "escalated": False, "conversation_history": []}
print(classifier_node(test_state))

{'query': 'My refund is not processed yet, I want to talk to manager', 'escalated': False, 'conversation_history': [], 'intent': 'escalation', 'confidence': 0.9, 'sentiment': 'negative'}


This cell tests the `classifier_node` with a sample query, demonstrating how it extracts intent, confidence, and sentiment from the input.

In [ ]:
def billing_node(state: AgentState) -> AgentState:
    response = llm.invoke(
        f"""You are a Billing Support Agent. Answer this customer query professionally.

Query: {state['query']}

Handle topics like payments, refunds, invoices, subscriptions."""
    )
    state["response"] = response.content
    return state


def technical_node(state: AgentState) -> AgentState:
    response = llm.invoke(
        f"""You are a Technical Support Agent. Help troubleshoot this issue.

Query: {state['query']}

Give clear step-by-step troubleshooting if needed."""
    )
    state["response"] = response.content
    return state


def faq_node(state: AgentState) -> AgentState:
    response = llm.invoke(
        f"""You are a General FAQ Agent. Answer this general question clearly and briefly.

Query: {state['query']}"""
    )
    state["response"] = response.content
    return state

This cell defines three specialized agent nodes: `billing_node`, `technical_node`, and `faq_node`. Each node uses the LLM to generate a professional response based on its specific domain and updates the agent's state with the generated response.

In [ ]:
def escalation_node(state: AgentState) -> AgentState:
    state["response"] = (
        " Your query has been escalated to a human agent. "
        "A support representative will contact you shortly. "
        f"(Reason: {state.get('sentiment', 'unknown')} sentiment / low confidence detected)"
    )
    state["escalated"] = True
    return state

This cell defines the `escalation_node`, which sets a generic escalation message and marks the query as `escalated` in the agent's state. This node is triggered for queries requiring human intervention.

In [ ]:
def route_query(state: AgentState) -> str:
    if state["confidence"] <= 0.6:
        return "escalation"
    return state["intent"]

This cell defines the `route_query` function, which determines the next node in the graph based on the classification confidence. If confidence is below or equal to 0.6, it routes to 'escalation'; otherwise, it routes to the detected 'intent'.

In [ ]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AgentState)

# Add nodes
graph.add_node("classifier", classifier_node)
graph.add_node("billing", billing_node)
graph.add_node("technical", technical_node)
graph.add_node("faq", faq_node)
graph.add_node("escalation", escalation_node)

# Entry point
graph.set_entry_point("classifier")

# Conditional edge — route_query decides where to go after classification
graph.add_conditional_edges(
    "classifier",
    route_query,
    {
        "billing": "billing",
        "technical": "technical",
        "faq": "faq",
        "escalation": "escalation"
    }
)

# Add edges to END for specialist nodes
graph.add_edge("billing", END)
graph.add_edge("technical", END)
graph.add_edge("faq", END)
graph.add_edge("escalation", END)

# Compile
app = graph.compile()

This cell constructs the `StateGraph` using `langgraph`. It adds all the defined nodes (`classifier`, `billing`, `technical`, `faq`, `escalation`), sets the entry point, defines conditional edges based on `route_query`, and adds direct edges from specialist nodes to `END`. Finally, it compiles the graph into an executable `app`.

In [ ]:
result = app.invoke({
    "query": "How do I reset my password?",
    "escalated": False,
    "conversation_history": []
})

print("Intent:", result["intent"])
print("Escalated:", result["escalated"])
print("Response:", result["response"])

Intent: technical
Escalated: False
Response: Resetting your password is a straightforward process. Here are the steps to follow:

**Method 1: Reset Password using the Forgot Password Option**

1. Go to the login page of the website or application you're trying to access.
2. Click on the "Forgot Password" or "Reset Password" link, usually located below the login form.
3. Enter your username or email address associated with your account.
4. Click on the "Reset Password" or "Send Reset Link" button.
5. Check your email inbox for a password reset email from the website or application.
6. Open the email and click on the password reset link.
7. Enter a new password and confirm it by re-entering it in the required field.
8. Click on the "Reset Password" or "Save Changes" button to save your new password.

**Method 2: Reset Password using Account Settings**

1. Log in to your account using your current password (if you remember it).
2. Go to your account settings or profile page.
3. Look for t

This cell invokes the compiled `app` with a sample 'technical' query ("How do I reset my password?") and prints the resulting intent, escalation status, and the generated response from the technical agent.

In [ ]:
result2 = app.invoke({
    "query": "This is the third time I'm asking! I want a refund NOW or I'm taking legal action!",
    "escalated": False,
    "conversation_history": []
})

print("Intent:", result2["intent"])
print("Sentiment:", result2["sentiment"])
print("Escalated:", result2["escalated"])
print("Response:", result2["response"])

Intent: escalation
Sentiment: negative
Escalated: True
Response:  Your query has been escalated to a human agent. A support representative will contact you shortly. (Reason: negative sentiment / low confidence detected)


This cell invokes the `app` with a sample 'escalation' query, testing the escalation mechanism based on keywords and negative sentiment. It prints the intent, sentiment, escalation status, and the escalation response.

In [ ]:
result3 = app.invoke({
    "query": "hmm okay thanks I guess",
    "escalated": False,
    "conversation_history": []
})

print("Intent:", result3["intent"])
print("Confidence:", result3["confidence"])
print("Escalated:", result3["escalated"])

Intent: faq
Confidence: 0.4
Escalated: True


This cell invokes the `app` with a query that has low confidence, demonstrating how the `route_query` function directs it to escalation even if the initial intent is 'faq'. It prints the intent, confidence, and escalation status.

In [ ]:
!pip install -q streamlit pyngrok

This cell installs `streamlit` and `pyngrok`, which are used to build and expose a web application.

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))

This cell sets the `ngrok` authentication token, retrieved from Colab's secrets, which is required to create a public URL for the Streamlit app.

In [ ]:
%%writefile app.py

import streamlit as st
import os
from typing import TypedDict, Optional, List, Literal
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

# ---------- Config ----------
os.environ["GROQ_API_KEY"] = st.secrets["GROQ_API_KEY"]

st.set_page_config(page_title="Customer Support AI")

# ---------- State ----------
class AgentState(TypedDict):
    query: str
    intent: Optional[str]
    confidence: Optional[float]
    sentiment: Optional[str]
    response: Optional[str]
    escalated: bool
    conversation_history: List[str]

class ClassificationResult(BaseModel):
    intent: Literal["billing", "technical", "faq", "escalation"] = Field(description="Query category")
    confidence: float = Field(description="0 to 1 confidence score")
    sentiment: Literal["positive", "neutral", "negative"] = Field(description="User's tone")
    reasoning: str = Field(description="One line reason")

# ---------- LLM ----------
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
structured_llm = llm.with_structured_output(ClassificationResult)

# ---------- Nodes ----------
def classifier_node(state: AgentState) -> AgentState:
    result = structured_llm.invoke(f"""Classify this customer support query.
Query: {state['query']}
Rules:
- If user mentions \"manager\", \"refund now\", \"legal\", \"cancel subscription\" -> escalation
- billing = payments, invoices, subscriptions, refunds
- technical = bugs, errors, how-to, troubleshooting
- faq = general info questions""")
    state["intent"] = result.intent
    state["confidence"] = result.confidence
    state["sentiment"] = result.sentiment
    return state

def billing_node(state: AgentState) -> AgentState:
    response = llm.invoke(f"You are a Billing Support Agent. Answer professionally.\nQuery: {state['query']}")
    state["response"] = response.content
    return state

def technical_node(state: AgentState) -> AgentState:
    response = llm.invoke(f"You are a Technical Support Agent. Give step-by-step help.\nQuery: {state['query']}")
    state["response"] = response.content
    return state

def faq_node(state: AgentState) -> AgentState:
    response = llm.invoke(f"You are a General FAQ Agent. Answer clearly and briefly.\nQuery: {state['query']}")
    state["response"] = response.content
    return state

def escalation_node(state: AgentState) -> AgentState:
    state["response"] = (
        "Your query has been escalated to a human agent. "
        "A support representative will contact you shortly. "
        f"(Reason: {state.get('sentiment','unknown')} sentiment / low confidence detected)"
    )
    state["escalated"] = True
    return state

def route_query(state: AgentState) -> str:
    if state["confidence"] <= 0.6:
        return "escalation"
    return state["intent"]

# ---------- Graph ----------
@st.cache_resource
def build_graph():
    graph = StateGraph(AgentState)
    graph.add_node("classifier", classifier_node)
    graph.add_node("billing", billing_node)
    graph.add_node("technical", technical_node)
    graph.add_node("faq", faq_node)
    graph.add_node("escalation", escalation_node)
    graph.set_entry_point("classifier")
    graph.add_conditional_edges("classifier", route_query, {
        "billing": "billing", "technical": "technical",
        "faq": "faq", "escalation": "escalation"
    })
    graph.add_edge("billing", END)
    graph.add_edge("technical", END)
    graph.add_edge("faq", END)
    graph.add_edge("escalation", END)
    return graph.compile()

app_graph = build_graph()

# ---------- UI ----------
st.title("Customer Support Multi-Agent System")
st.caption("LangGraph | Conditional Routing + Escalation")

if "history" not in st.session_state:
    st.session_state.history = []

query = st.text_area("Enter your query:", height=100)

if st.button("Submit") and query.strip():
    with st.spinner("Processing..."):
        result = app_graph.invoke({
            "query": query,
            "escalated": False,
            "conversation_history": []
        })
    st.session_state.history.append(result)

for r in reversed(st.session_state.history):
    st.markdown("---")
    st.markdown(f"**Query:** {r['query']}")
    col1, col2, col3 = st.columns(3)
    col1.metric("Intent", r["intent"])
    col2.metric("Confidence", f"{r['confidence']:.2f}")
    col3.metric("Sentiment", r["sentiment"])

    if r["escalated"]:
        st.error("Escalated to Human Agent")
    st.info(r["response"])


Overwriting app.py


This cell writes the Streamlit application code to a file named `app.py`. This code defines the Streamlit UI, initializes the LangGraph agent, and handles user interactions for the customer support bot.

In [ ]:
import os
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/secrets.toml", "w") as f:
    f.write(f'GROQ_API_KEY = "{userdata.get("GROQ_API_KEY")}"\n')

This cell creates a `.streamlit` directory and writes the `GROQ_API_KEY` to a `secrets.toml` file, allowing the Streamlit application to access the API key securely.

In [ ]:
!pkill -f streamlit
!pkill -f ngrok

This cell stops any running Streamlit or ngrok processes, ensuring a clean restart for the web application.

In [ ]:
import time
time.sleep(3)

from pyngrok import ngrok
ngrok.kill()

public_url = ngrok.connect(8501)
print("APP URL:", public_url)

import subprocess
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
time.sleep(6)

APP URL: NgrokTunnel: "https://postnasal-angled-sessions.ngrok-free.dev" -> "http://localhost:8501"


This cell initiates the Streamlit application and exposes it via ngrok. It waits for a few seconds, kills any previous ngrok tunnels, connects to a new ngrok tunnel for port 8501 (where Streamlit runs), prints the public URL, and then starts the Streamlit server in a subprocess.